In [59]:
import sys
import os

# Get project root (one level above notebook/)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: /Users/suriyaa/Documents/Projects/LLMOps/Flipkart Product Recommender


In [60]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from flipkart.config import Config
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
model = ChatGroq(model='openai/gpt-oss-120b')
embedding_model = Config.EMBEDDING_MODEL
embeddings = HuggingFaceEndpointEmbeddings(model=embedding_model)

In [61]:
import os
from langchain_astradb import AstraDBVectorStore
from dotenv import load_dotenv

load_dotenv(override=True)

vstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="flipkart_database",
    api_endpoint=os.getenv('ASTRA_DB_API_ENDPOINT'),
    token=os.getenv('ASTRA_DB_APPLICATION_TOKEN'),
    namespace=Config.ASTRA_DB_KEYSPACE
)



In [62]:
retriever = vstore.as_retriever()

# RAG without History

In [83]:
prompt = ChatPromptTemplate.from_messages([

    ('system', """

        You are an expert assistant. Use only the provided context. to answer the user query
        Guidelines:
        1. Use ONLY the information from the context above.
        2. If multiple pieces of information are relevant, combine them logically.
        3. If the answer is not explicitly stated, say:
        "The provided context does not contain sufficient information."
        4. Do NOT use prior knowledge.
        5. Keep the answer structured and precise.

        Context: {context}
    """),

    ('human',"""
    
        Question: {question}
        Answer: 
    """)



])

In [84]:
chain = ({
    "context": retriever,
    "question": RunnablePassthrough()
}
| prompt
| model
| StrOutputParser()
)

In [85]:
chain.invoke('Best earphones')

'**Earphones highlighted as “best” in the provided context**\n\n| Product (as listed) | Description indicating it is a top choice |\n|----------------------|--------------------------------------------|\n| OnePlus Bullets Wireless Z **Bass Edition** Bluetooth Headset | Described as “Best earphone.” |\n| BoAt BassHeads 100 Wired Headset | Called “Best budgeted earphones… Best earphones till used by me.” |\n| realme Buds 2 Wired Headset | Noted for “nice earphones… the build quality is best… better for bass lover.” |\n| OnePlus Bullets Wireless Z Bluetooth Headset | Referred to as “Really great earphones… I totally recommend this.” |\n\nThese four products are identified in the context as among the best earphones.'

'**Earphones highlighted as “best” in the provided context**\n\n| Product (as listed) | Description indicating it is a top choice |\n|----------------------|--------------------------------------------|\n| OnePlus Bullets Wireless Z **Bass Edition** Bluetooth Headset | Described as “Best earphone.” |\n| BoAt BassHeads 100 Wired Headset | Called “Best budgeted earphones… Best earphones till used by me.” |\n| realme Buds 2 Wired Headset | Noted for “nice earphones… the build quality is best… better for bass lover.” |\n| OnePlus Bullets Wireless Z Bluetooth Headset | Referred to as “Really great earphones… I totally recommend this.” |\n\nThese four products are identified in the context as among the best earphones.'

# RAG with History

In [19]:
context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and user question, rewrite it as a standalone question."),
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{input}")  
])

In [63]:
qa_prompt = ChatPromptTemplate.from_messages([
            ("system", """You're an e-commerce bot answering product-related queries using reviews and titles.
                          Stick to context. Be concise and helpful.\n\nCONTEXT:\n{context}\n\nQUESTION: {input}"""),
            MessagesPlaceholder(variable_name="chat_history"), 
            ("human", "{input}")  
        ])

In [79]:
from langchain_core.output_parsers import StrOutputParser
history_aware_retriever = (
    context_prompt 
    | llm 
    | StrOutputParser()
    | retriever
)

In [65]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

question_answer_chain = qa_prompt | llm


In [ ]:
from langchain_core.runnables import RunnableWith

rag_chain = (
    {
        "context": history_aware_retriever,
        "input": RunnablePassthrough(),
        "chat_history": RunnablePassthrough()
    }
    | qa_prompt
    | llm
)



In [74]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
history_store = {}
def get_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in history_store:
        history_store[session_id] = ChatMessageHistory()
    return history_store[session_id]

In [76]:
from langchain_community.chat_message_histories import ChatMessageHistory
history_store = {}
history_store['test_user'] = ChatMessageHistory()

In [80]:
response = rag_chain.invoke({'input': 'Give me some names of good speakers', 'chat_history': history_store['test_user'].messages})

AttributeError: 'AIMessage' object has no attribute 'replace'